# 15 — Paket review buta p100 (CPU)
Input: output notebook 13. Output: zip review <=45 MB per bagian + manifest SHA256. Tidak menyertakan review_key_PRIVATE.json. Tanpa GPU, tanpa token.


In [ ]:
from pathlib import Path
import zipfile, hashlib, json
# CPU only. Reads notebook 13 output (attached as input) and packs ONLY the blind review folder
# into <=45 MB zip parts so it can be downloaded. The private key is never packed.
INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/review_pack')
OUT.mkdir(parents=True, exist_ok=False)
hits = [p.parent for p in INPUT.rglob('index.html') if p.parent.name == 'review' and p.parent.parent.name == 'p100']
if len(hits) != 1: raise ValueError(f'Expected one p100/review folder; found {hits}')
review = hits[0]
files = sorted(p for p in review.rglob('*') if p.is_file())
if any('PRIVATE' in p.name for p in files): raise ValueError('private key inside review folder')
if not (review/'review.csv').exists(): raise ValueError('review.csv missing')
LIMIT = 45 * 2**20
parts, current, size = [], [], 0
for p in files:
    s = p.stat().st_size
    if current and size + s > LIMIT:
        parts.append(current); current, size = [], 0
    current.append(p); size += s
parts.append(current)
manifest = dict(source=str(review), n_files=len(files), files={}, parts={})
for i, part in enumerate(parts):
    target = OUT/f'review_part{i:02d}.zip'
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_STORED) as z:
        for p in part:
            rel = 'review/' + str(p.relative_to(review))
            z.write(p, rel)
            manifest['files'][rel] = hashlib.sha256(p.read_bytes()).hexdigest()
    manifest['parts'][target.name] = dict(n=len(part), bytes=target.stat().st_size,
                                          sha256=hashlib.sha256(target.read_bytes()).hexdigest())
(OUT/'review_pack_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(len(files), 'files;', {k: round(v['bytes']/2**20, 1) for k, v in manifest['parts'].items()}, 'MB')
